# Testing adata concatenation

## Download

In [ ]:
from pathlib import Path

import anndata as ad

from storage import download_from_r2, fetch_uploaded_r2_keys

In [ ]:
keys = fetch_uploaded_r2_keys(
    prefix="arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/"
)

In [ ]:
keys_download = list(keys)[:2]
keys_download = [
    "arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/SRX13061245.h5ad",
    "arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/SRX10048396.h5ad",
]

In [ ]:
for k in keys_download:
    name = Path(k).stem
    download_from_r2(
        r2_key=k,
        local_path=Path().resolve().parents[1] / "tmp" / f"{name}.h5ad",
        verify_md5=True,
    )

## Load adata objects and concatenate

In [ ]:
a1 = ad.read_h5ad(Path().resolve().parents[1] / "tmp" / f"{Path(keys_download[0]).stem}.h5ad")
a2 = ad.read_h5ad(Path().resolve().parents[1] / "tmp" / f"{Path(keys_download[1]).stem}.h5ad")

In [ ]:
ac = ad.concat([a1, a2], label="batch", keys=["study1", "study2"])

## Inspect how a1 / a2 differ from ac

In [ ]:
def summarize(name, a):
    print(f"=== {name} ===")
    print(f"shape        : {a.shape}")
    print(f"X dtype/type : {a.X.dtype if a.X is not None else None} / {type(a.X).__name__}")
    print(f"obs cols     : {list(a.obs.columns)}")
    print(f"var cols     : {list(a.var.columns)}")
    print(f"layers       : {list(a.layers.keys())}")
    print(f"obsm         : {list(a.obsm.keys())}")
    print(f"varm         : {list(a.varm.keys())}")
    print(f"obsp         : {list(a.obsp.keys())}")
    print(f"varp         : {list(a.varp.keys())}")
    print(f"uns          : {list(a.uns.keys())}")
    print(f"raw          : {a.raw is not None}")
    print()


summarize("a1", a1)
summarize("a2", a2)
summarize("ac", ac)

In [ ]:
v1 = set(a1.var_names)
v2 = set(a2.var_names)
vc = set(ac.var_names)

print(f"a1 n_vars           : {len(v1)}")
print(f"a2 n_vars           : {len(v2)}")
print(f"ac n_vars           : {len(vc)}")
print(f"intersection(a1,a2) : {len(v1 & v2)}")
print(f"union(a1,a2)        : {len(v1 | v2)}")
print(f"ac == intersection  : {vc == (v1 & v2)}  (join='inner' default)")
print(f"only in a1 (dropped): {len(v1 - v2)}")
print(f"only in a2 (dropped): {len(v2 - v1)}")
print(f"ac var order == sorted intersection order preserved from a1: "
      f"{list(ac.var_names) == [v for v in a1.var_names if v in v2]}")

In [ ]:
import numpy as np
from scipy import sparse


def to_dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)


shared_vars = list(ac.var_names)

for name, src in [("study1", a1), ("study2", a2)]:
    src_sub = src[:, shared_vars]
    ac_sub = ac[ac.obs["batch"] == name]

    x_src = to_dense(src_sub.X)
    x_ac = to_dense(ac_sub.X)

    same_shape = x_src.shape == x_ac.shape
    identical = same_shape and np.array_equal(x_src, x_ac)
    print(f"{name}: src{x_src.shape} vs ac{x_ac.shape} | X identical: {identical}")
    if same_shape and not identical:
        print(f"  max abs diff: {np.abs(x_src - x_ac).max()}")

In [ ]:
print("var columns dropped by concat (default merge=None):")
print(f"  a1.var cols : {list(a1.var.columns)}")
print(f"  a2.var cols : {list(a2.var.columns)}")
print(f"  ac.var cols : {list(ac.var.columns)}  <- empty unless merge=... passed")
print()
print("uns dropped by concat (default uns_merge=None):")
print(f"  a1.uns keys : {list(a1.uns.keys())}")
print(f"  ac.uns keys : {list(ac.uns.keys())}")
print()
print("layers kept only if present in all objects:")
print(f"  a1.layers   : {list(a1.layers.keys())}")
print(f"  a2.layers   : {list(a2.layers.keys())}")
print(f"  ac.layers   : {list(ac.layers.keys())}")
print()
print("obs gained 'batch' column from label=... :")
print(f"  ac.obs['batch'] value counts:\n{ac.obs['batch'].value_counts()}")

In [ ]:
a2.var

# Roundtrip Test

In [ ]:
from pathlib import Path
import anndata as ad
from h5ad_concat import H5adConcatConfig, run_h5ad_concat

cfg = H5adConcatConfig(
    datasetsPath=Path().resolve().parents[1] / "tests/datasets_sample10.csv",
    uploadAtlas=True,
    atlasR2Key="lung/atlas_test.h5ad",
    minPctCellsAfterQc=0.5,
    minCellsPerGene=0,
)

result = run_h5ad_concat(cfg)
print(result.nObs, result.studiesSeen, result.skipped)

In [ ]:
cfg.model_dump()

In [ ]:
result.model_dump()

In [ ]:
import anndata as ad
atlas = ad.read_h5ad(result.outputPath)

In [ ]:
(result.nVars == atlas.n_vars == atlas.var.shape[0]) & (result.nObs == atlas.n_obs == atlas.obs.shape[0])

In [ ]:
atlas.obs.groupby("study_accession")["SRX_accession"].value_counts()

In [ ]:
import numpy as np
from scipy import sparse


def to_dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)


shared = list(atlas.var_names)

for srx, src in [("SRX13061245", a1), ("SRX10048396", a2)]:
    atlas_sub = atlas[atlas.obs["SRX_accession"] == srx]
    src_sub = src[:, shared]

    x_atlas = to_dense(atlas_sub.X)
    x_src = to_dense(src_sub.X)

    same_shape = x_atlas.shape == x_src.shape
    identical = same_shape and np.array_equal(x_atlas, x_src)
    print(f"{srx}: atlas{x_atlas.shape} vs src{x_src.shape} | X identical: {identical}")
    if same_shape and not identical:
        print(f"  max abs diff: {np.abs(x_atlas - x_src).max()}")